In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
pip uninstall -y torchao


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -q -U transformers accelerate peft bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [6]:
# smart_mcq_full_pipeline.py
# Three models -> OOF stacking -> weighted ensemble -> final submission
# Model A (scratch): TF-IDF + feedforward net
# Model B (pretrained, LoRA fine-tuned): microsoft/deberta-v3-base via AutoModelForMultipleChoice
# Model C (additional/zero-shot): Qwen3-8B next-token logit scoring
# Ensemble: OOF logits from A+B blended via grid search, C blended in as a prior on top

import os, re, json, ast, ast as _ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

LETTERS = ["A", "B", "C", "D", "E"]
letter2idx = {l: i for i, l in enumerate(LETTERS)}
train_df["label"] = train_df["answer"].map(letter2idx)

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(kf.split(train_df)):
    train_df.loc[val_idx, "fold"] = fold


# =====================================================================
# MODEL A - from scratch: TF-IDF cosine-sim features -> small feedforward net
# =====================================================================
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def build_tfidf_features(df, vectorizer=None, fit=False):
    prompts = df["prompt"].fillna("").map(clean_text).tolist()
    opts_flat = []
    for l in LETTERS:
        opts_flat += df[l].fillna("").map(clean_text).tolist()

    if fit:
        vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
        vectorizer.fit(prompts + opts_flat)

    p_vec = vectorizer.transform(prompts)
    feats = np.zeros((len(df), 5), dtype=np.float32)
    for i, l in enumerate(LETTERS):
        o_vec = vectorizer.transform(df[l].fillna("").map(clean_text).tolist())
        num = np.asarray(p_vec.multiply(o_vec).sum(axis=1)).ravel()
        p_norm = np.sqrt(np.asarray(p_vec.multiply(p_vec).sum(axis=1)).ravel())
        o_norm = np.sqrt(np.asarray(o_vec.multiply(o_vec).sum(axis=1)).ravel())
        feats[:, i] = num / (p_norm * o_norm + 1e-8)
    return feats, vectorizer

class FFN(nn.Module):
    def __init__(self, in_dim=5, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 5),
        )
    def forward(self, x):
        return self.net(x)

def train_ffn(X, y, epochs=60, lr=1e-3):
    model = FFN(X.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss()
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        out = model(Xt)
        loss = lossf(out, yt)
        loss.backward()
        opt.step()
    return model

full_feats, tfidf_vec = build_tfidf_features(train_df, fit=True)
test_feats_a, _ = build_tfidf_features(test_df, vectorizer=tfidf_vec, fit=False)

oof_logits_a = np.zeros((len(train_df), 5), dtype=np.float32)
test_logits_a = np.zeros((len(test_df), 5), dtype=np.float32)

for fold in range(N_FOLDS):
    tr_idx = train_df[train_df.fold != fold].index
    va_idx = train_df[train_df.fold == fold].index
    m = train_ffn(full_feats[tr_idx], train_df.loc[tr_idx, "label"].values)
    m.eval()
    with torch.no_grad():
        oof_logits_a[va_idx] = m(torch.tensor(full_feats[va_idx], dtype=torch.float32)).numpy()
        test_logits_a += m(torch.tensor(test_feats_a, dtype=torch.float32)).numpy() / N_FOLDS

print("Model A (scratch TF-IDF+FFN) done.")


# =====================================================================
# MODEL B - pretrained, LoRA fine-tuned: deberta-v3-base AutoModelForMultipleChoice
# =====================================================================
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_B_ID = "microsoft/deberta-v3-base"
tok_b = AutoTokenizer.from_pretrained(MODEL_B_ID)

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, has_label=True, max_len=256):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.has_label = has_label
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row["prompt"])
        opts = [str(row[l]) for l in LETTERS]
        first = [prompt] * 5
        enc = self.tok(first, opts, truncation=True, max_length=self.max_len, padding="max_length")
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.has_label:
            item["labels"] = torch.tensor(row["label"], dtype=torch.long)
        return item

def collate_mcq(batch):
    out = {}
    keys = batch[0].keys()
    for k in keys:
        if k == "labels":
            out[k] = torch.stack([b[k] for b in batch])
        else:
            out[k] = torch.stack([b[k] for b in batch])
    return out

def build_lora_model():
    base = AutoModelForMultipleChoice.from_pretrained(MODEL_B_ID)
    base = base.float()  # force fp32 base, no ambiguity from any default torch_dtype

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8, lora_alpha=16, lora_dropout=0.1,
        target_modules=["query_proj", "value_proj"],
    )
    model = get_peft_model(base, lora_cfg, autocast_adapter_dtype=False)

    # belt-and-suspenders: force every trainable param (LoRA + classifier head) to fp32
    for p in model.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()

    model.print_trainable_parameters()
    return model

# TrainingArguments across transformers versions dropped group_by_length in some builds
import inspect
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
ta_kwargs = dict(
    output_dir="./tmp_b",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="no",
    report_to=[],
)
if "group_by_length" in ta_sig:
    ta_kwargs["group_by_length"] = True

oof_logits_b = np.zeros((len(train_df), 5), dtype=np.float32)
test_logits_b = np.zeros((len(test_df), 5), dtype=np.float32)
test_ds_b = MCQDataset(test_df, tok_b, has_label=False)

for fold in range(N_FOLDS):
    tr_df = train_df[train_df.fold != fold]
    va_df = train_df[train_df.fold == fold]
    tr_ds = MCQDataset(tr_df, tok_b)
    va_ds = MCQDataset(va_df, tok_b)

    model_b = build_lora_model()
    args = TrainingArguments(**ta_kwargs)
    trainer = Trainer(model=model_b, args=args, train_dataset=tr_ds, data_collator=collate_mcq)
    trainer.train()

    va_pred = trainer.predict(va_ds)
    oof_logits_b[va_df.index] = va_pred.predictions

    test_pred = trainer.predict(test_ds_b)
    test_logits_b += test_pred.predictions / N_FOLDS

print("Model B (LoRA DeBERTa) done.")


# =====================================================================
# MODEL C - additional model of choice: Qwen3-8B zero-shot next-token scoring
# =====================================================================
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

QWEN_ID = "Qwen/Qwen3-8B"
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tok_c = AutoTokenizer.from_pretrained(QWEN_ID)
tok_c.padding_side = "left"
if tok_c.pad_token is None:
    tok_c.pad_token = tok_c.eos_token

model_c = AutoModelForCausalLM.from_pretrained(
    QWEN_ID, quantization_config=bnb_cfg, device_map="auto", torch_dtype=torch.float16
)
model_c.eval()

letter_ids = {}
for l in LETTERS:
    cand_space = tok_c.encode(" " + l, add_special_tokens=False)
    cand = tok_c.encode(l, add_special_tokens=False)
    letter_ids[l] = cand_space[0] if len(cand_space) == 1 else cand[0]
letter_id_list = [letter_ids[l] for l in LETTERS]

def build_prompt_c(row):
    opts = "\n".join(f"{l}) {row[l]}" for l in LETTERS)
    msg = [{"role": "user", "content": f"Question: {row['prompt']}\n\n{opts}\n\nPick the single best answer. Reply with only the letter."}]
    text = tok_c.apply_chat_template(msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return text + "Answer:"

def get_qwen_logits(df, batch_size=8):
    prompts = [build_prompt_c(r) for _, r in df.iterrows()]
    logits_out = np.zeros((len(df), 5), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i+batch_size]
            enc = tok_c(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model_c.device)
            out = model_c(**enc)
            last_logits = out.logits[:, -1, :]
            logits_out[i:i+batch_size] = last_logits[:, letter_id_list].float().cpu().numpy()
    return logits_out

oof_logits_c = get_qwen_logits(train_df)   # zero-shot, no folds needed (no training)
test_logits_c = get_qwen_logits(test_df)

print("Model C (Qwen3-8B zero-shot) done.")


# =====================================================================
# ENSEMBLE: grid search blend weights on OOF logits, MAP@3 as the objective
# =====================================================================
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def map3(y_true, probs):
    top3 = np.argsort(probs, axis=1)[:, ::-1][:, :3]
    scores = []
    for t, preds in zip(y_true, top3):
        if t == preds[0]:
            scores.append(1.0)
        elif t == preds[1]:
            scores.append(1/2)
        elif t == preds[2]:
            scores.append(1/3)
        else:
            scores.append(0.0)
    return np.mean(scores)

y_true = train_df["label"].values
pa, pb, pc = softmax(oof_logits_a), softmax(oof_logits_b), softmax(oof_logits_c)

best_score, best_w = -1, (1, 0, 0)
for wa in np.arange(0, 1.05, 0.05):
    for wb in np.arange(0, 1.05 - wa, 0.05):
        wc = 1 - wa - wb
        blend = wa * pa + wb * pb + wc * pc
        s = map3(y_true, blend)
        if s > best_score:
            best_score, best_w = s, (wa, wb, wc)

print(f"Best OOF MAP@3: {best_score:.4f} at weights (A,B,C) = {best_w}")

wa, wb, wc = best_w
test_blend = wa * softmax(test_logits_a) + wb * softmax(test_logits_b) + wc * softmax(test_logits_c)
top3_idx = np.argsort(test_blend, axis=1)[:, ::-1][:, :3]
preds = [" ".join(LETTERS[j] for j in row) for row in top3_idx]

sub = pd.DataFrame({"ID": test_df["id"], "Prediction": preds})
sub.to_csv("submission.csv", index=False)
print(sub.head())

Model A (scratch TF-IDF+FFN) done.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias   

trainable params: 295,681 || all params: 184,718,594 || trainable%: 0.1601


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model C (Qwen3-8B zero-shot) done.
Best OOF MAP@3: 0.9350 at weights (A,B,C) = (np.float64(0.2), np.float64(0.65), np.float64(0.15000000000000002))
   ID Prediction
0   1      A B D
1   2      B C E
2   3      B E C
3   4      E C D
4   5      C D A
